# 🐼 Pandas Practice — Advanced Exercises (11–20)

Continuation of the practice series covering advanced pandas features.

| Exercise | Topic |
|----------|-------|
| 11 | Reshaping — melt, stack, unstack |
| 12 | MultiIndex |
| 13 | Categorical Data |
| 14 | Method Chaining — assign() & pipe() |
| 15 | eval() & query() |
| 16 | shift() & diff() — Lag Analysis |
| 17 | explode() — List Columns |
| 18 | Memory Optimization |
| 19 | Sampling & Bootstrap |
| 20 | RFM Customer Segmentation (Mini Project) |

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)
print('pandas:', pd.__version__)

---
## Exercise 11 — Reshaping: melt, stack, unstack

**Scenario:** Transform a wide quarterly sales report into long format and back.

Tasks:
1. Use `melt()` to convert the wide format into long format with columns `region`, `quarter`, `sales`
2. Use `pivot()` to convert back to wide format
3. Use `stack()` on a MultiIndex DataFrame to get a Series
4. Use `unstack()` to convert back to DataFrame
5. Find the quarter with the highest average sales across all regions

In [ ]:
wide = pd.DataFrame({
    'region': ['North', 'South', 'East', 'West'],
    'Q1': [120000, 95000, 140000, 110000],
    'Q2': [130000, 98000, 155000, 118000],
    'Q3': [145000, 102000, 160000, 125000],
    'Q4': [160000, 115000, 175000, 138000],
})
print('Wide format:')
print(wide)

# 1. melt to long format
# YOUR CODE HERE

# 2. pivot back to wide
# YOUR CODE HERE

# 3. stack example
df_mi = wide.set_index('region')
# YOUR CODE HERE (stack df_mi)

# 4. unstack
# YOUR CODE HERE

# 5. Best quarter by avg sales
# YOUR CODE HERE

---
## Exercise 12 — MultiIndex

**Scenario:** Work with a hierarchical index for a stock portfolio.

Tasks:
1. Create a MultiIndex DataFrame with `(ticker, date)` as the index
2. Select all rows for ticker `'AAPL'`
3. Select `AAPL` on `2024-01-03`
4. Use `xs()` to cross-section by date across all tickers
5. Reset the index and group by `ticker` to get mean `close` price

In [ ]:
tickers = ['AAPL', 'GOOG', 'MSFT']
dates = pd.date_range('2024-01-01', periods=5, freq='B')
idx = pd.MultiIndex.from_product([tickers, dates], names=['ticker', 'date'])
np.random.seed(1)
portfolio = pd.DataFrame({
    'open':  np.random.uniform(100, 400, len(idx)).round(2),
    'close': np.random.uniform(100, 400, len(idx)).round(2),
    'volume': np.random.randint(1_000_000, 50_000_000, len(idx))
}, index=idx)
print(portfolio.head(8))

# 2. All AAPL rows
# YOUR CODE HERE

# 3. AAPL on 2024-01-03
# YOUR CODE HERE

# 4. Cross-section by date 2024-01-02
# YOUR CODE HERE

# 5. Reset index + mean close per ticker
# YOUR CODE HERE

---
## Exercise 13 — Categorical Data

**Scenario:** Work with ordered survey responses to analyse satisfaction levels.

Tasks:
1. Convert `rating` to an ordered Categorical: `Very Poor < Poor < Neutral < Good < Excellent`
2. Count responses per rating (respects order)
3. Filter for ratings `>= Good`
4. Add a `score` column: Very Poor=1, Poor=2, Neutral=3, Good=4, Excellent=5
5. Check memory savings vs plain object dtype

In [ ]:
np.random.seed(5)
n = 500
survey = pd.DataFrame({
    'respondent_id': range(1, n+1),
    'dept': np.random.choice(['Eng','Sales','HR','Finance'], n),
    'rating': np.random.choice(['Very Poor','Poor','Neutral','Good','Excellent'], n,
                               p=[0.05, 0.10, 0.25, 0.40, 0.20]),
})

# 1. Ordered Categorical
order = ['Very Poor', 'Poor', 'Neutral', 'Good', 'Excellent']
# YOUR CODE HERE

# 2. Value counts (ordered)
# YOUR CODE HERE

# 3. Filter >= Good
# YOUR CODE HERE

# 4. Numeric score column
score_map = {'Very Poor':1, 'Poor':2, 'Neutral':3, 'Good':4, 'Excellent':5}
# YOUR CODE HERE

# 5. Memory comparison
# YOUR CODE HERE

---
## Exercise 14 — Method Chaining with assign() & pipe()

**Scenario:** Build a clean, readable ETL pipeline using method chaining.

Tasks:
1. In a single chain: filter active users, add `full_name`, add `age_group` (bins: 18-30, 31-45, 46+), sort by `ltv` desc
2. Write a `normalize_ltv(df)` function and apply it with `pipe()`
3. Use `assign()` to add `revenue_per_day = ltv / tenure_days` in the chain
4. Chain `query()` to keep only users with `revenue_per_day > 1`
5. Print the final shape and first 5 rows

In [ ]:
np.random.seed(42)
n = 300
users = pd.DataFrame({
    'user_id': range(1, n+1),
    'first': np.random.choice(['Alice','Bob','Carol','Dave','Eve'], n),
    'last':  np.random.choice(['Smith','Jones','Lee','Brown','Taylor'], n),
    'age':   np.random.randint(18, 65, n),
    'active': np.random.choice([True, False], n, p=[0.8, 0.2]),
    'ltv':    np.random.exponential(500, n).round(2),
    'tenure_days': np.random.randint(1, 1000, n),
})

# 2. pipe helper
def normalize_ltv(df):
    df = df.copy()
    df['ltv_normalized'] = (df['ltv'] - df['ltv'].min()) / (df['ltv'].max() - df['ltv'].min())
    return df

# 1, 3, 4 — full chain
result = (
    users
    # YOUR CODE HERE: filter active
    # YOUR CODE HERE: assign full_name, age_group, revenue_per_day
    # YOUR CODE HERE: pipe normalize_ltv
    # YOUR CODE HERE: query revenue_per_day > 1
    # YOUR CODE HERE: sort by ltv desc
)

# 5.
print('Shape:', result.shape)
print(result[['full_name','age_group','ltv','revenue_per_day','ltv_normalized']].head())

---
## Exercise 15 — eval() & query()

**Scenario:** Analyse financial data efficiently using pandas expressions.

Tasks:
1. Use `query()` to filter rows where `pe_ratio < 20` AND `market_cap > 10`
2. Use `query()` with a Python variable: filter where `dividend_yield > threshold`
3. Use `eval()` to compute `enterprise_value = market_cap + total_debt - cash`
4. Use `eval()` to compute `ev_ebitda = enterprise_value / ebitda` in one expression
5. Compare performance: `eval()` vs standard pandas for adding two columns on 500K rows

In [ ]:
np.random.seed(3)
n = 200
stocks = pd.DataFrame({
    'ticker':         [f'TKR{i:03d}' for i in range(n)],
    'market_cap':     np.random.uniform(1, 500, n).round(2),   # billions
    'pe_ratio':       np.random.uniform(5, 60, n).round(1),
    'dividend_yield': np.random.uniform(0, 8, n).round(2),
    'total_debt':     np.random.uniform(0, 100, n).round(2),
    'cash':           np.random.uniform(0, 50, n).round(2),
    'ebitda':         np.random.uniform(0.5, 50, n).round(2),
})

# 1. query: pe < 20 AND market_cap > 10
# YOUR CODE HERE

# 2. query with variable
threshold = 3.0
# YOUR CODE HERE

# 3. eval enterprise_value
# YOUR CODE HERE

# 4. eval ev_ebitda (chain or use inplace)
# YOUR CODE HERE

# 5. Performance comparison on large df
import time
big = pd.DataFrame({'a': np.random.randn(500_000), 'b': np.random.randn(500_000)})
t0 = time.time(); _ = big['a'] + big['b']; t_std = time.time() - t0
t0 = time.time(); _ = big.eval('a + b');    t_ev  = time.time() - t0
print(f'Standard: {t_std*1000:.2f}ms  |  eval: {t_ev*1000:.2f}ms')

---
## Exercise 16 — shift() & diff() — Lag Analysis

**Scenario:** Analyse month-over-month and year-over-year changes in revenue.

Tasks:
1. Add `prev_month_revenue` using `shift(1)`
2. Add `mom_change` (month-over-month absolute change) using `diff()`
3. Add `mom_pct` (month-over-month % change)
4. Add `yoy_change` by shifting 12 periods
5. Flag months where revenue dropped more than 10% MoM

In [ ]:
months = pd.date_range('2022-01-01', periods=36, freq='MS')
np.random.seed(9)
base = 100_000
trend = np.linspace(0, 30000, 36)
seasonal = 15000 * np.sin(np.linspace(0, 4*np.pi, 36))
noise = np.random.normal(0, 5000, 36)
revenue_data = pd.DataFrame({
    'month': months,
    'revenue': (base + trend + seasonal + noise).round(0).astype(int)
})
revenue_data = revenue_data.set_index('month')
print(revenue_data.head())

# 1. prev_month_revenue
# YOUR CODE HERE

# 2. mom_change
# YOUR CODE HERE

# 3. mom_pct
# YOUR CODE HERE

# 4. yoy_change (shift 12)
# YOUR CODE HERE

# 5. Flag drops > 10% MoM
# YOUR CODE HERE

print(revenue_data.tail(10))

---
## Exercise 17 — explode() with List Columns

**Scenario:** A product database stores multiple tags and categories per item as lists.

Tasks:
1. Use `explode()` to expand the `tags` list column into one row per tag
2. Count how many products have each tag
3. Find the top 5 most common tags
4. Explode `sizes` and count available size options per category
5. Find products that have both `'wireless'` AND `'bluetooth'` tags

In [ ]:
products = pd.DataFrame({
    'product_id': [1, 2, 3, 4, 5, 6],
    'name': ['Headphones','Keyboard','Mouse','Speaker','Webcam','Monitor'],
    'category': ['Audio','Input','Input','Audio','Video','Video'],
    'price': [79.99, 49.99, 29.99, 99.99, 59.99, 299.99],
    'tags': [
        ['wireless','bluetooth','noise-cancelling'],
        ['mechanical','rgb','wireless'],
        ['wireless','ergonomic','silent'],
        ['bluetooth','waterproof','portable'],
        ['hd','usb','plug-and-play'],
        ['4k','hdr','usb-c','bluetooth'],
    ],
    'sizes': [['S','M','L'],['M','L'],['S','M'],['M','L','XL'],['M'],['L','XL']]
})

# 1. explode tags
# YOUR CODE HERE

# 2. count per tag
# YOUR CODE HERE

# 3. top 5 tags
# YOUR CODE HERE

# 4. explode sizes + count per category
# YOUR CODE HERE

# 5. products with both 'wireless' AND 'bluetooth'
# YOUR CODE HERE

---
## Exercise 18 — Memory Optimization

**Scenario:** A 500K row dataset is consuming too much RAM — optimize it.

Tasks:
1. Check memory usage per column with `memory_usage(deep=True)`
2. Downcast integer columns to the smallest type that fits (`pd.to_numeric` with `downcast`)
3. Downcast float columns similarly
4. Convert low-cardinality string columns to `category` dtype
5. Report total memory before and after, and the % reduction

In [ ]:
np.random.seed(42)
n = 500_000
df = pd.DataFrame({
    'user_id':   np.random.randint(0, 100_000, n),         # int64
    'age':       np.random.randint(18, 90, n),              # int64 but fits int8
    'score':     np.random.uniform(0, 100, n),              # float64 but float32 ok
    'rating':    np.random.randint(1, 6, n),                # int64 but fits int8
    'country':   np.random.choice(['US','UK','DE','JP','BR'], n),  # object
    'plan':      np.random.choice(['free','basic','pro','enterprise'], n),  # object
    'revenue':   np.random.exponential(100, n),             # float64
})

mem_before = df.memory_usage(deep=True).sum() / 1024**2
print(f'Memory before: {mem_before:.2f} MB')
print(df.dtypes)

# 2. Downcast integers
# YOUR CODE HERE

# 3. Downcast floats
# YOUR CODE HERE

# 4. Convert string cols to category
# YOUR CODE HERE

# 5. Report savings
# YOUR CODE HERE

---
## Exercise 19 — Sampling & Bootstrap Confidence Intervals

**Scenario:** Estimate the true average order value with confidence intervals using bootstrap resampling.

Tasks:
1. Draw a random sample of 200 rows (without replacement) — set `random_state=42`
2. Draw a stratified sample: 50 rows per `channel`, proportional to channel size
3. Bootstrap the mean of `order_value` — resample with replacement 1000 times, store each mean
4. Compute the 95% confidence interval from the bootstrap distribution
5. Compare the sample mean to the true population mean

In [ ]:
np.random.seed(42)
n = 10_000
orders = pd.DataFrame({
    'order_id':    range(n),
    'channel':     np.random.choice(['Web','Mobile','In-Store','Phone'], n, p=[0.5,0.3,0.15,0.05]),
    'order_value': np.random.exponential(75, n).round(2),
    'region':      np.random.choice(['North','South','East','West'], n),
})
print('Population mean order value:', orders['order_value'].mean().round(2))

# 1. Simple random sample
# YOUR CODE HERE

# 2. Stratified sample: 50 per channel
# YOUR CODE HERE

# 3. Bootstrap 1000 iterations
# YOUR CODE HERE

# 4. 95% CI
# YOUR CODE HERE

# 5. Compare
# YOUR CODE HERE

---
## Exercise 20 — Mini Project: RFM Customer Segmentation

**Scenario:** Segment customers using the classic **RFM model** (Recency, Frequency, Monetary).

RFM scores each customer 1–4 on three dimensions:
- **Recency**: how recently they bought (4 = most recent)
- **Frequency**: how often they buy (4 = most frequent)
- **Monetary**: how much they spend (4 = highest spender)

Tasks:
1. Compute RFM metrics per customer: last purchase date, number of orders, total spend
2. Score each metric 1–4 using `pd.qcut()` (quartiles)
3. Create `rfm_score` = concatenation of R, F, M scores as a string (e.g. `'444'`)
4. Map scores to segments: Champions (444,443,434), Loyal (334,343,344), At Risk (244,234,224), Lost (111,112,121,122)
5. Count customers per segment and calculate avg total spend per segment
6. Print a summary table sorted by avg spend desc

In [ ]:
from datetime import date

np.random.seed(42)
n_customers = 500
n_orders = 3000
snapshot_date = pd.Timestamp('2024-12-31')

transactions = pd.DataFrame({
    'customer_id': np.random.choice([f'C{i:04d}' for i in range(n_customers)], n_orders),
    'order_date':  pd.to_datetime(np.random.choice(
        pd.date_range('2023-01-01', '2024-12-31').astype(str), n_orders
    )),
    'order_value': np.random.exponential(80, n_orders).round(2)
})

# 1. RFM metrics
# YOUR CODE HERE
# rfm = transactions.groupby('customer_id').agg(...)

# 2. Score 1-4 using qcut
# Recency: lower days = better = score 4
# Frequency & Monetary: higher = better = score 4
# YOUR CODE HERE

# 3. rfm_score string
# YOUR CODE HERE

# 4. Segment mapping
champions   = ['444','443','434','344']
loyal       = ['334','343','333','324','342']
at_risk     = ['244','234','224','243','242']
lost        = ['111','112','121','122','211','212']

def map_segment(score):
    if score in champions: return 'Champions'
    if score in loyal:     return 'Loyal'
    if score in at_risk:   return 'At Risk'
    if score in lost:      return 'Lost'
    return 'Others'

# YOUR CODE HERE (apply map_segment)

# 5 & 6. Summary
# YOUR CODE HERE

---
# ✅ Solutions (11–20)

> Try first!

## Solution 11 — Reshaping

In [ ]:
wide = pd.DataFrame({'region':['North','South','East','West'],'Q1':[120000,95000,140000,110000],'Q2':[130000,98000,155000,118000],'Q3':[145000,102000,160000,125000],'Q4':[160000,115000,175000,138000]})
long = wide.melt(id_vars='region', var_name='quarter', value_name='sales')
print('Long:'); print(long.head(6))
wide2 = long.pivot(index='region', columns='quarter', values='sales')
print('Wide back:'); print(wide2)
stacked = wide.set_index('region').stack()
print('Stacked:'); print(stacked.head(6))
unstacked = stacked.unstack()
print('Unstacked:'); print(unstacked)
best_q = long.groupby('quarter')['sales'].mean().idxmax()
print('Best quarter:', best_q)

## Solution 12 — MultiIndex

In [ ]:
tickers = ['AAPL','GOOG','MSFT']
dates = pd.date_range('2024-01-01', periods=5, freq='B')
idx = pd.MultiIndex.from_product([tickers, dates], names=['ticker','date'])
np.random.seed(1)
portfolio = pd.DataFrame({'open':np.random.uniform(100,400,len(idx)).round(2),'close':np.random.uniform(100,400,len(idx)).round(2),'volume':np.random.randint(1_000_000,50_000_000,len(idx))}, index=idx)
print(portfolio.loc['AAPL'])
print(portfolio.loc[('AAPL', '2024-01-03')])
print(portfolio.xs('2024-01-02', level='date'))
print(portfolio.reset_index().groupby('ticker')['close'].mean().round(2))

## Solution 13 — Categorical Data

In [ ]:
np.random.seed(5)
n = 500
survey = pd.DataFrame({'respondent_id':range(1,n+1),'dept':np.random.choice(['Eng','Sales','HR','Finance'],n),'rating':np.random.choice(['Very Poor','Poor','Neutral','Good','Excellent'],n,p=[0.05,0.10,0.25,0.40,0.20])})
order = ['Very Poor','Poor','Neutral','Good','Excellent']
survey['rating'] = pd.Categorical(survey['rating'], categories=order, ordered=True)
print(survey['rating'].value_counts().sort_index())
print(survey[survey['rating'] >= 'Good'].shape)
score_map = {'Very Poor':1,'Poor':2,'Neutral':3,'Good':4,'Excellent':5}
survey['score'] = survey['rating'].map(score_map)
obj_mem = survey['rating'].astype(str).memory_usage(deep=True)
cat_mem = survey['rating'].memory_usage(deep=True)
print(f'Object: {obj_mem} bytes | Category: {cat_mem} bytes | Saved: {obj_mem-cat_mem} bytes')

## Solution 14 — Method Chaining

In [ ]:
np.random.seed(42)
n = 300
users = pd.DataFrame({'user_id':range(1,n+1),'first':np.random.choice(['Alice','Bob','Carol','Dave','Eve'],n),'last':np.random.choice(['Smith','Jones','Lee','Brown','Taylor'],n),'age':np.random.randint(18,65,n),'active':np.random.choice([True,False],n,p=[0.8,0.2]),'ltv':np.random.exponential(500,n).round(2),'tenure_days':np.random.randint(1,1000,n)})
def normalize_ltv(df):
    df = df.copy()
    df['ltv_normalized'] = (df['ltv'] - df['ltv'].min()) / (df['ltv'].max() - df['ltv'].min())
    return df
result = (
    users
    .query('active == True')
    .assign(
        full_name = lambda d: d['first'] + ' ' + d['last'],
        age_group = lambda d: pd.cut(d['age'], bins=[17,30,45,100], labels=['18-30','31-45','46+']),
        revenue_per_day = lambda d: (d['ltv'] / d['tenure_days']).round(4)
    )
    .pipe(normalize_ltv)
    .query('revenue_per_day > 1')
    .sort_values('ltv', ascending=False)
)
print('Shape:', result.shape)
print(result[['full_name','age_group','ltv','revenue_per_day','ltv_normalized']].head())

## Solution 15 — eval() & query()

In [ ]:
np.random.seed(3)
n = 200
stocks = pd.DataFrame({'ticker':[f'TKR{i:03d}' for i in range(n)],'market_cap':np.random.uniform(1,500,n).round(2),'pe_ratio':np.random.uniform(5,60,n).round(1),'dividend_yield':np.random.uniform(0,8,n).round(2),'total_debt':np.random.uniform(0,100,n).round(2),'cash':np.random.uniform(0,50,n).round(2),'ebitda':np.random.uniform(0.5,50,n).round(2)})
print(stocks.query('pe_ratio < 20 and market_cap > 10').shape)
threshold = 3.0
print(stocks.query('dividend_yield > @threshold').shape)
stocks.eval('enterprise_value = market_cap + total_debt - cash', inplace=True)
stocks.eval('ev_ebitda = enterprise_value / ebitda', inplace=True)
print(stocks[['ticker','enterprise_value','ev_ebitda']].head())
import time
big = pd.DataFrame({'a':np.random.randn(500_000),'b':np.random.randn(500_000)})
t0 = time.time(); _ = big['a']+big['b']; print(f'Standard: {(time.time()-t0)*1000:.2f}ms')
t0 = time.time(); _ = big.eval('a+b');   print(f'eval:     {(time.time()-t0)*1000:.2f}ms')

## Solution 16 — shift() & diff()

In [ ]:
months = pd.date_range('2022-01-01', periods=36, freq='MS')
np.random.seed(9)
base=100_000; trend=np.linspace(0,30000,36); seasonal=15000*np.sin(np.linspace(0,4*np.pi,36)); noise=np.random.normal(0,5000,36)
revenue_data = pd.DataFrame({'revenue':(base+trend+seasonal+noise).round(0).astype(int)}, index=months)
revenue_data['prev_month_revenue'] = revenue_data['revenue'].shift(1)
revenue_data['mom_change'] = revenue_data['revenue'].diff()
revenue_data['mom_pct'] = revenue_data['revenue'].pct_change().round(4)
revenue_data['yoy_change'] = revenue_data['revenue'].diff(12)
revenue_data['big_drop'] = revenue_data['mom_pct'] < -0.10
print(revenue_data.tail(10))
print('Big drops:', revenue_data['big_drop'].sum())

## Solution 17 — explode()

In [ ]:
products = pd.DataFrame({'product_id':[1,2,3,4,5,6],'name':['Headphones','Keyboard','Mouse','Speaker','Webcam','Monitor'],'category':['Audio','Input','Input','Audio','Video','Video'],'price':[79.99,49.99,29.99,99.99,59.99,299.99],'tags':[['wireless','bluetooth','noise-cancelling'],['mechanical','rgb','wireless'],['wireless','ergonomic','silent'],['bluetooth','waterproof','portable'],['hd','usb','plug-and-play'],['4k','hdr','usb-c','bluetooth']],'sizes':[['S','M','L'],['M','L'],['S','M'],['M','L','XL'],['M'],['L','XL']]})
exploded_tags = products.explode('tags')
print(exploded_tags[['name','tags']].head(8))
tag_counts = exploded_tags['tags'].value_counts()
print('Top 5 tags:')
print(tag_counts.head(5))
sizes_by_cat = products.explode('sizes').groupby('category')['sizes'].count()
print('Size options per category:')
print(sizes_by_cat)
has_wireless = set(products[products['tags'].apply(lambda t: 'wireless' in t)]['product_id'])
has_bluetooth = set(products[products['tags'].apply(lambda t: 'bluetooth' in t)]['product_id'])
both = has_wireless & has_bluetooth
print('Both wireless & bluetooth:', products[products['product_id'].isin(both)]['name'].tolist())

## Solution 18 — Memory Optimization

In [ ]:
np.random.seed(42)
n = 500_000
df = pd.DataFrame({'user_id':np.random.randint(0,100_000,n),'age':np.random.randint(18,90,n),'score':np.random.uniform(0,100,n),'rating':np.random.randint(1,6,n),'country':np.random.choice(['US','UK','DE','JP','BR'],n),'plan':np.random.choice(['free','basic','pro','enterprise'],n),'revenue':np.random.exponential(100,n)})
mem_before = df.memory_usage(deep=True).sum() / 1024**2
for col in ['user_id','age','rating']:
    df[col] = pd.to_numeric(df[col], downcast='integer')
for col in ['score','revenue']:
    df[col] = pd.to_numeric(df[col], downcast='float')
for col in ['country','plan']:
    df[col] = df[col].astype('category')
mem_after = df.memory_usage(deep=True).sum() / 1024**2
print(f'Before: {mem_before:.2f} MB')
print(f'After:  {mem_after:.2f} MB')
print(f'Saved:  {mem_before-mem_after:.2f} MB ({(1-mem_after/mem_before)*100:.1f}% reduction)')
print(df.dtypes)

## Solution 19 — Sampling & Bootstrap

In [ ]:
np.random.seed(42)
n = 10_000
orders = pd.DataFrame({'order_id':range(n),'channel':np.random.choice(['Web','Mobile','In-Store','Phone'],n,p=[0.5,0.3,0.15,0.05]),'order_value':np.random.exponential(75,n).round(2),'region':np.random.choice(['North','South','East','West'],n)})
pop_mean = orders['order_value'].mean()
sample = orders.sample(200, random_state=42)
print('Sample mean:', sample['order_value'].mean().round(2), '| Pop mean:', pop_mean.round(2))
strat = orders.groupby('channel', group_keys=False).apply(lambda x: x.sample(50, random_state=42))
print('Stratified sample shape:', strat.shape)
boot_means = [orders['order_value'].sample(200, replace=True).mean() for _ in range(1000)]
boot_series = pd.Series(boot_means)
ci_low  = boot_series.quantile(0.025).round(2)
ci_high = boot_series.quantile(0.975).round(2)
print(f'Bootstrap mean: {boot_series.mean():.2f}')
print(f'95% CI: [{ci_low}, {ci_high}]')
print(f'True population mean inside CI: {ci_low <= pop_mean <= ci_high}')

## Solution 20 — RFM Customer Segmentation

In [ ]:
np.random.seed(42)
n_customers=500; n_orders=3000
snapshot_date = pd.Timestamp('2024-12-31')
transactions = pd.DataFrame({'customer_id':np.random.choice([f'C{i:04d}' for i in range(n_customers)],n_orders),'order_date':pd.to_datetime(np.random.choice(pd.date_range('2023-01-01','2024-12-31').astype(str),n_orders)),'order_value':np.random.exponential(80,n_orders).round(2)})

rfm = transactions.groupby('customer_id').agg(
    recency   = ('order_date', lambda x: (snapshot_date - x.max()).days),
    frequency = ('order_id',   'count') if 'order_id' in transactions else ('order_value','count'),
    monetary  = ('order_value','sum')
).round(2)

rfm['R'] = pd.qcut(rfm['recency'],   q=4, labels=[4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'],  q=4, labels=[1,2,3,4]).astype(int)
rfm['rfm_score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)

champions=['444','443','434','344']; loyal=['334','343','333','324','342']
at_risk=['244','234','224','243','242']; lost=['111','112','121','122','211','212']
def map_segment(s):
    if s in champions: return 'Champions'
    if s in loyal:     return 'Loyal'
    if s in at_risk:   return 'At Risk'
    if s in lost:      return 'Lost'
    return 'Others'
rfm['segment'] = rfm['rfm_score'].apply(map_segment)

summary = rfm.groupby('segment').agg(
    customers=('monetary','count'),
    avg_spend=('monetary','mean'),
    avg_frequency=('frequency','mean')
).round(2).sort_values('avg_spend', ascending=False)
print('RFM Segment Summary:')
print(summary)